# Quadratic-Bezier interrupted-run inspection — read only

Use this notebook only after the fixed comparison was interrupted. It mounts Google Drive, hashes and inspects the preserved output state without opening images, revealing metrics, deleting files, renaming directories, resuming work, or starting a new comparison. Run all four code cells and return the downloaded `quadratic_bezier_incomplete_diagnostic.json`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
OUTPUT_PARENT = Path('/content/drive/MyDrive/latent-stroke-dynamics-rgb')
OUTPUT_DIR = OUTPUT_PARENT / 'quadratic-bezier-fixed-comparison-v1'
INCOMPLETE_DIR = OUTPUT_PARENT / 'quadratic-bezier-fixed-comparison-v1.incomplete'
print('CELL 1 COMPLETE — DRIVE MOUNTED FOR READ-ONLY INSPECTION')
print('completed directory exists:', OUTPUT_DIR.is_dir())
print('incomplete directory exists:', INCOMPLETE_DIR.is_dir())

In [ ]:
from hashlib import sha256
import json
from pathlib import Path

EXPECTED_RUN_COUNT = 36

def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def inspect_root(root: Path) -> dict:
    if not root.is_dir():
        return {'exists': False}
    files = sorted(path for path in root.rglob('*') if path.is_file())
    runs_root = root / 'runs'
    run_dirs = (
        sorted(path for path in runs_root.glob('*/seed_*/*') if path.is_dir())
        if runs_root.is_dir()
        else []
    )
    completed_units = []
    partial_units = []
    completed_integrity = []
    for run_dir in run_dirs:
        relative_unit = str(run_dir.relative_to(runs_root))
        summary_path = run_dir / 'summary.json'
        summary_sha_path = run_dir / 'summary.sha256'
        if not summary_path.is_file():
            partial_units.append(relative_unit)
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        recorded_summary_sha = (
            summary_sha_path.read_text(encoding='utf-8').strip()
            if summary_sha_path.is_file()
            else None
        )
        observed_summary_sha = file_sha256(summary_path)
        artifact_manifest = summary.get('artifact_sha256')
        artifact_manifest_ok = isinstance(artifact_manifest, dict)
        verified_artifact_count = 0
        if artifact_manifest_ok:
            for relative_path, expected_hash in artifact_manifest.items():
                artifact_path = run_dir / relative_path
                if not artifact_path.is_file() or file_sha256(artifact_path) != expected_hash:
                    artifact_manifest_ok = False
                    break
                verified_artifact_count += 1
        integrity = {
            'status_complete': summary.get('status') == 'quadratic_bezier_comparison_run_complete',
            'summary_checksum_valid': recorded_summary_sha == observed_summary_sha,
            'artifact_manifest_valid': artifact_manifest_ok,
            'every_executed_stroke_improved': summary.get('every_executed_stroke_improved') is True,
            'best_not_worse_than_final': summary.get('best_not_worse_than_final') is True,
            'training_performed_false': summary.get('training_performed') is False,
            'learned_model_used_false': summary.get('learned_model_used') is False,
            'closed_experiments_changed_false': summary.get('closed_experiments_changed') is False,
        }
        completed_units.append(relative_unit)
        completed_integrity.append(
            {
                'unit': relative_unit,
                'all_checks_passed': all(integrity.values()),
                'checks': integrity,
                'verified_artifact_count': verified_artifact_count,
            }
        )
    aggregate_path = root / 'aggregate_summary.json'
    aggregate = {'exists': aggregate_path.is_file()}
    if aggregate_path.is_file():
        value = json.loads(aggregate_path.read_text(encoding='utf-8'))
        aggregate_sha_path = root / 'aggregate_summary.sha256'
        aggregate.update(
            {
                'status': value.get('status'),
                'completed_run_count': value.get('completed_run_count'),
                'completed_pair_count': value.get('completed_pair_count'),
                'integrity_passed': value.get('integrity_passed'),
                'checksum_valid': (
                    aggregate_sha_path.is_file()
                    and aggregate_sha_path.read_text(encoding='utf-8').strip() == file_sha256(aggregate_path)
                ),
            }
        )
    failure_path = root / 'failure.json'
    failure = {'exists': failure_path.is_file()}
    if failure_path.is_file():
        value = json.loads(failure_path.read_text(encoding='utf-8'))
        failure.update(
            {
                'status': value.get('status'),
                'error_type': value.get('error_type'),
                'error': value.get('error'),
            }
        )
    return {
        'exists': True,
        'file_count': len(files),
        'total_bytes': sum(path.stat().st_size for path in files),
        'discovered_run_directory_count': len(run_dirs),
        'completed_run_count': len(completed_units),
        'partial_run_count': len(partial_units),
        'not_yet_started_run_count': max(0, EXPECTED_RUN_COUNT - len(run_dirs)),
        'completed_units': completed_units,
        'partial_units': partial_units,
        'completed_unit_integrity': completed_integrity,
        'all_completed_units_valid': bool(completed_integrity) and all(
            item['all_checks_passed'] for item in completed_integrity
        ),
        'aggregate': aggregate,
        'failure': failure,
    }

print('CELL 2 COMPLETE — READ-ONLY HASH AND STRUCTURE CHECKS DEFINED')

In [ ]:
import json
from pathlib import Path

completed = inspect_root(OUTPUT_DIR)
incomplete = inspect_root(INCOMPLETE_DIR)
if completed.get('exists') and incomplete.get('exists'):
    state = 'ambiguous_both_completed_and_incomplete_exist'
    next_action = 'stop_and_return_diagnostic_without_modifying_either_directory'
elif completed.get('exists'):
    state = 'completed_directory_exists_after_interruption'
    next_action = 'stop_and_return_diagnostic_without_rerunning'
elif incomplete.get('exists'):
    state = 'preserved_incomplete_attempt_found'
    next_action = 'preserve_and_return_diagnostic_for_recovery_design'
else:
    state = 'no_preserved_output_directory_found'
    next_action = 'stop_and_return_diagnostic_before_any_new_execution'
diagnostic = {
    'status': 'quadratic_bezier_interrupted_run_inspected_read_only',
    'state': state,
    'next_action': next_action,
    'expected_run_count': EXPECTED_RUN_COUNT,
    'completed_output': completed,
    'incomplete_output': incomplete,
    'images_opened': False,
    'metrics_revealed': False,
    'files_deleted': False,
    'files_renamed': False,
    'execution_resumed': False,
    'new_execution_started': False,
}
DIAGNOSTIC_PATH = Path('/content/quadratic_bezier_incomplete_diagnostic.json')
DIAGNOSTIC_PATH.write_text(json.dumps(diagnostic, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('CELL 3 COMPLETE — INTERRUPTION STATE INSPECTED WITHOUT MUTATION')
print('state:', state)
print('completed runs in completed directory:', completed.get('completed_run_count', 0))
print('completed runs in incomplete directory:', incomplete.get('completed_run_count', 0))
print('partial runs in incomplete directory:', incomplete.get('partial_run_count', 0))
print('next action:', next_action)

In [ ]:
from google.colab import files

files.download(str(DIAGNOSTIC_PATH))
print('CELL 4 COMPLETE — DIAGNOSTIC DOWNLOAD STARTED')